# 🥘 SousChef AI v2 — Advanced RAG-Based Personalized Cooking Copilot

An advanced notebook-first AI project built around real **Food.com recipes + user interactions**.

### Core AI techniques
- Sentence-Transformer embeddings
- FAISS vector search with cached embeddings
- Cross-encoder reranking
- Semantic RAG over recipes + culinary knowledge
- Optional LLM grounded generation
- Bayesian recipe rating
- Collaborative filtering with matrix factorization
- Hybrid recommendation
- Cuisine classification model comparison
- Offline retrieval and RAG evaluation
- Interactive `ipywidgets` notebook dashboard

> **Kaggle inputs recommended**
> 1. `Food.com Recipes and Interactions` (`RAW_recipes.csv`, `RAW_interactions.csv`)
> 2. Optional: Kaggle **What's Cooking?** dataset (`train.json`) for cuisine classification.
>
> The notebook still runs without the cuisine dataset; only the cuisine classifier section will be skipped.

## 1. Project Introduction

### Problem

Most recipe recommenders rely on exact ingredient matching or popularity. **SousChef AI** combines semantic understanding, user behavior, recipe quality, dietary constraints, and retrieval-augmented generation.

### System

`ingredients / cooking goal / user profile → semantic retrieval → hybrid ranking → reranking → explanation`

For cooking questions:

`question → sentence embedding → FAISS knowledge retrieval → grounded local/LLM answer + evidence`

### Evaluation

The notebook evaluates:
- semantic retrieval with **HitRate@K** and **MRR**
- RAG retrieval with **Hit@K** and **MRR**
- cuisine classification with **Accuracy** and **Macro-F1**

In [1]:
# Kaggle/Jupyter setup.
# Kaggle already includes numpy/pandas/sklearn/scipy. These are the main extras.
!pip install -q sentence-transformers faiss-cpu ipywidgets openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import re
import ast
import glob
import json
import math
import hashlib
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
)
from sklearn.decomposition import TruncatedSVD

import matplotlib.pyplot as plt
import plotly.express as px

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42

# Start with 20K while developing. Increase to 50K+ for the final Kaggle version.
MAX_RECIPES = 20_000
MAX_INTERACTIONS = 500_000
RAG_RECIPE_CHUNKS = 3_000

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CACHE_DIR = Path("/kaggle/working/souschef_cache") if Path("/kaggle/working").exists() else Path("./souschef_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Cache directory:", CACHE_DIR)

Cache directory: /kaggle/working/souschef_cache


## 2. Dataset Loading

The loader searches both local folders and Kaggle `/kaggle/input`.

For Food.com it uses:
- `RAW_recipes.csv`
- `RAW_interactions.csv`

Only the columns needed for collaborative filtering are loaded from interactions to avoid wasting memory on the full review text.

In [3]:
def find_file(patterns):
    search_roots = [
        Path("."),
        Path.cwd(),
        Path.cwd().parent,
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    matches = []
    for root in search_roots:
        if root.exists():
            for pattern in patterns:
                matches.extend(glob.glob(str(root / "**" / pattern), recursive=True))

    matches = [
        m for m in matches
        if ".ipynb_checkpoints" not in m
        and "/notebooks/" not in m.lower()
    ]
    return matches[0] if matches else None


def safe_parse_list(x):
    if isinstance(x, list):
        return x
    if x is None or (not isinstance(x, (list, tuple)) and pd.isna(x)):
        return []
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        return [i.strip() for i in x.split(",") if i.strip()]
    return list(x) if isinstance(x, (tuple, set)) else []


def build_fallback_recipe_data():
    print("⚠ No external recipe dataset found — using tiny fallback data.")
    return pd.DataFrame([
        {
            "recipe_id": 1,
            "name": "Tomato Garlic Pasta",
            "minutes": 25,
            "ingredients": "['pasta','tomato','garlic','olive oil','basil','salt']",
            "steps": "['boil pasta','cook garlic','add tomato','combine']",
            "tags": "['vegetarian','italian','quick']",
        },
        {
            "recipe_id": 2,
            "name": "Chickpea Curry",
            "minutes": 35,
            "ingredients": "['chickpeas','onion','tomato','garlic','ginger','cumin','coriander']",
            "steps": "['saute onion','add spices','add tomato','simmer chickpeas']",
            "tags": "['vegetarian','indian','high protein']",
        },
        {
            "recipe_id": 3,
            "name": "Vegetable Fried Rice",
            "minutes": 20,
            "ingredients": "['rice','carrot','peas','soy sauce','garlic','spring onion']",
            "steps": "['stir fry vegetables','add rice','season']",
            "tags": "['vegetarian','asian','quick']",
        },
    ])


def load_data(max_recipes=MAX_RECIPES, max_interactions=MAX_INTERACTIONS):
    recipe_file = find_file(["RAW_recipes.csv", "recipes.csv", "sample_recipes.csv"])
    interaction_file = find_file(["RAW_interactions.csv", "interactions.csv", "reviews.csv"])

    print("Recipe file:", recipe_file)
    print("Interaction file:", interaction_file)

    if recipe_file is None:
        recipes = build_fallback_recipe_data()
    else:
        recipes = pd.read_csv(recipe_file, nrows=max_recipes)

    if "id" in recipes.columns and "recipe_id" not in recipes.columns:
        recipes = recipes.rename(columns={"id": "recipe_id"})

    if "recipe_id" not in recipes.columns:
        recipes["recipe_id"] = np.arange(len(recipes))

    for col in ["name", "ingredients", "steps", "tags"]:
        if col not in recipes.columns:
            recipes[col] = ""

    if "minutes" not in recipes.columns:
        recipes["minutes"] = np.nan

    interactions = pd.DataFrame(columns=["user_id", "recipe_id", "rating", "date"])

    if interaction_file is not None:
        try:
            header = pd.read_csv(interaction_file, nrows=0).columns.tolist()
            desired = [c for c in ["user_id", "recipe_id", "rating", "date"] if c in header]
            interactions = pd.read_csv(
                interaction_file,
                usecols=desired,
                nrows=max_interactions
            )

            if "rating" in interactions.columns:
                interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce")
        except Exception as e:
            print("Could not load interactions:", e)

    # Aggregate real Food.com ratings into recipe table.
    if len(interactions) and {"recipe_id", "rating"}.issubset(interactions.columns):
        rated = interactions[interactions["rating"] > 0].copy()
        agg = (
            rated.groupby("recipe_id")
            .agg(rating=("rating", "mean"), review_count=("rating", "size"))
            .reset_index()
        )
        recipes = recipes.drop(columns=["rating", "review_count"], errors="ignore")
        recipes = recipes.merge(agg, on="recipe_id", how="left")
    else:
        if "rating" not in recipes.columns:
            recipes["rating"] = np.nan
        if "review_count" not in recipes.columns:
            recipes["review_count"] = 0

    recipes["rating"] = pd.to_numeric(recipes["rating"], errors="coerce").fillna(0)
    recipes["review_count"] = pd.to_numeric(recipes["review_count"], errors="coerce").fillna(0)
    recipes["minutes"] = pd.to_numeric(recipes["minutes"], errors="coerce")

    return recipes, interactions


recipes, interactions = load_data()

print("\nRecipes:", recipes.shape)
print("Interactions:", interactions.shape)
recipes.head(3)

Recipe file: /kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/RAW_recipes.csv
Interaction file: /kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/RAW_interactions.csv

Recipes: (20000, 14)
Interactions: (500000, 4)


,name,recipe_id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,rating,review_count
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , ...","autumn is my favorite time of year to cook! this recipe \r\ncan be prepared either spicy or sweet, your choice!\r\ntwo of my posted mexi...","['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']",7,0.000000,0.0
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'breakfast',...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough into the bottom and sides of a 12 inch pizza pan', 'bake for 5 minutes until set but not ...",this recipe calls for the crust to be prebaked a bit before adding ingredients. feel free to change sausage to ham or bacon. this warms ...,"['prepared pizza crust', 'sausage patty', 'eggs', 'milk', 'salt and pepper', 'cheese']",6,4.666667,3.0
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'main-dish', 'chili', 'crock-pot-slow-cooker', 'dietary', 'equipment', '4-hours-or-less']","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add chopped onions to ground beef when almost brown and sautee until wilted', 'add all other ingredi...",this modified version of 'mom's' chili was a hit at our 2004 christmas party. we made an extra large pot to have some left to freeze but...,"['ground beef', 'yellow onions', 'diced tomatoes', 'tomato paste', 'tomato soup', 'rotel tomatoes', 'kidney beans', 'water', 'chili powd...",13,0.000000,0.0


## 3. Data Cleaning

This section:
- parses Food.com stringified ingredient/tag/step lists
- normalizes text
- builds dietary heuristic fields
- removes impossible cooking-time values from ranking
- creates a **Bayesian weighted rating** so a recipe with one 5-star review does not automatically outrank a recipe with thousands of strong reviews

In [4]:
def normalize_text(text):
    if text is None or (not isinstance(text, (list, tuple)) and pd.isna(text)):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_list_items(items):
    items = safe_parse_list(items)
    cleaned = []
    for item in items:
        x = normalize_text(item)
        if x:
            cleaned.append(x)
    return cleaned


MEAT_TERMS = set(
    "chicken beef pork lamb turkey bacon sausage ham fish shrimp tuna salmon anchovy meat duck".split()
)
DAIRY_EGG_TERMS = set(
    "milk cheese butter cream yogurt egg paneer parmesan feta mozzarella ghee".split()
)
GLUTEN_TERMS = set(
    "wheat flour bread pasta tortilla noodles couscous barley semolina".split()
)


def contains_any(text, terms):
    text = normalize_text(text)
    return any(re.search(r"\b" + re.escape(t) + r"\b", text) for t in terms)


def bayesian_rating(rating, count, global_mean, prior_count):
    return (
        (count / (count + prior_count)) * rating
        + (prior_count / (count + prior_count)) * global_mean
    )


def add_clean_features(df):
    df = df.copy()

    df["ingredients_list"] = df["ingredients"].apply(clean_list_items)
    df["steps_list"] = df["steps"].apply(clean_list_items)
    df["tags_list"] = df["tags"].apply(clean_list_items)

    df["ingredient_text"] = df["ingredients_list"].apply(" ".join)
    df["step_text"] = df["steps_list"].apply(" ".join)
    df["tag_text"] = df["tags_list"].apply(" ".join)

    df["recipe_text"] = (
        df["name"].fillna("").astype(str)
        + " ingredients "
        + df["ingredient_text"]
        + " tags "
        + df["tag_text"]
        + " instructions "
        + df["step_text"]
    ).apply(normalize_text)

    df["is_vegetarian"] = ~df["ingredient_text"].apply(lambda x: contains_any(x, MEAT_TERMS))
    df["is_vegan"] = (
        df["is_vegetarian"]
        & ~df["ingredient_text"].apply(lambda x: contains_any(x, DAIRY_EGG_TERMS))
    )
    df["is_gluten_free"] = ~df["ingredient_text"].apply(lambda x: contains_any(x, GLUTEN_TERMS))

    df["ingredient_count"] = df["ingredients_list"].apply(len)
    df["step_count"] = df["steps_list"].apply(len)

    # Treat extreme/invalid times as missing for recommendation purposes.
    clean_minutes = df["minutes"].where(df["minutes"].between(1, 1440))
    median_minutes = clean_minutes.median() if clean_minutes.notna().any() else 30
    df["minutes_clean"] = clean_minutes.fillna(median_minutes)

    rated = df[df["rating"] > 0]
    global_mean = rated["rating"].mean() if len(rated) else 4.0
    positive_counts = df.loc[df["review_count"] > 0, "review_count"]
    prior_count = float(positive_counts.quantile(0.75)) if len(positive_counts) else 10.0
    prior_count = max(prior_count, 10.0)

    df["bayesian_rating"] = bayesian_rating(
        df["rating"],
        df["review_count"],
        global_mean,
        prior_count
    )

    if "cuisine" not in df.columns:
        df["cuisine"] = "unknown"

    return df


recipes = add_clean_features(recipes)

print("Clean recipes:", recipes.shape)
print("Mean rating prior:", round(recipes.loc[recipes.rating > 0, "rating"].mean(), 3))
recipes[
    [
        "recipe_id",
        "name",
        "minutes_clean",
        "rating",
        "review_count",
        "bayesian_rating",
        "is_vegetarian",
        "is_vegan",
    ]
].head()

Clean recipes: (20000, 29)
Mean rating prior: 4.58


,recipe_id,name,minutes_clean,rating,review_count,bayesian_rating,is_vegetarian,is_vegan
0,137739,arriba baked winter squash mexican style,55.0,0.000000,0.0,4.579801,True,False
1,31490,a bit different breakfast pizza,30.0,4.666667,3.0,4.599847,False,False
2,112140,all in the kitchen chili,130.0,0.000000,0.0,4.579801,False,False
3,59389,alouette potatoes,45.0,4.500000,2.0,4.566501,True,False
4,44061,amish tomato ketchup for canning,190.0,0.000000,0.0,4.579801,True,True


## 4. Exploratory Data Analysis

The point of this EDA is not just visualization. It checks whether the ranking signals are sensible:
- cooking-time distribution
- most frequent ingredients
- ratings vs review volume
- dietary coverage

In [5]:
def top_ingredients(df, top_n=20):
    all_ings = []
    for items in df["ingredients_list"]:
        all_ings.extend(items)
    s = pd.Series(all_ings).value_counts().head(top_n)
    return s.rename_axis("ingredient").reset_index(name="count")


print("Total recipes:", f"{len(recipes):,}")
print("Median cooking time:", round(recipes["minutes_clean"].median(), 1), "minutes")
print("Recipes with ratings:", f"{int((recipes.review_count > 0).sum()):,}")
print("Vegetarian (heuristic):", f"{int(recipes.is_vegetarian.sum()):,}")
print("Vegan (heuristic):", f"{int(recipes.is_vegan.sum()):,}")

fig = px.histogram(
    recipes[recipes["minutes_clean"].between(1, 180)],
    x="minutes_clean",
    nbins=45,
    title="Cooking Time Distribution"
)
fig.show()

top_ing = top_ingredients(recipes, 20)
px.bar(top_ing, x="ingredient", y="count", title="Top 20 Ingredients").show()

rated_plot = recipes[(recipes["review_count"] > 0) & (recipes["rating"] > 0)].copy()
if len(rated_plot):
    px.scatter(
        rated_plot.sample(min(len(rated_plot), 5000), random_state=RANDOM_STATE),
        x="review_count",
        y="rating",
        hover_name="name",
        log_x=True,
        title="Raw Rating vs Review Count"
    ).show()

Total recipes: 20,000
Median cooking time: 40.0 minutes
Recipes with ratings: 8,529
Vegetarian (heuristic): 13,071
Vegan (heuristic): 4,398


## 5. Ingredient Processing

User input is normalized into ingredient tokens. The recommender uses:
- ingredient coverage
- missing ingredient count
- hard exclusions for allergies/dislikes

> Dietary/allergy detection here is heuristic and **not a medical safety guarantee**.

In [6]:
def parse_user_ingredients(text):
    return [
        normalize_text(i)
        for i in re.split(r"[,\n]+", str(text))
        if normalize_text(i)
    ]


def ingredient_overlap(user_ingredients, recipe_ingredients):
    user_set = set(user_ingredients)
    recipe_set = set(recipe_ingredients)

    if not recipe_set:
        return 0.0, [], []

    matched = sorted(user_set.intersection(recipe_set))
    missing = sorted(recipe_set.difference(user_set))

    # Fraction of the recipe's ingredient list already available.
    score = len(matched) / max(len(recipe_set), 1)
    return score, matched, missing


def violates_exclusions(recipe_ingredients, excluded):
    if not excluded:
        return False
    text = " ".join(recipe_ingredients)
    return any(re.search(r"\b" + re.escape(x) + r"\b", text) for x in excluded)


# Quick test
ingredient_overlap(
    parse_user_ingredients("tomato, garlic, rice, onion"),
    recipes.iloc[0]["ingredients_list"]
)

(0.0,
 [],
 ['butter',
  'honey',
  'mexican seasoning',
  'mixed spice',
  'olive oil',
  'salt',
  'winter squash'])

## 6. Recipe Embeddings + FAISS Vector Index

The notebook encodes recipe text with `all-MiniLM-L6-v2`.

Important production upgrade: **embeddings and FAISS index are cached** under `/kaggle/working/souschef_cache`, so rerunning the notebook does not always require re-embedding every recipe.

In [7]:
class RecipeEmbeddingIndex:
    def __init__(self, df, model_name=EMBEDDING_MODEL, cache_dir=CACHE_DIR):
        self.df = df.reset_index(drop=True)
        self.texts = self.df["recipe_text"].tolist()
        self.model_name = model_name
        self.cache_dir = Path(cache_dir)

        signature_text = (
            str(len(self.df))
            + "|"
            + "|".join(self.df["recipe_id"].astype(str).head(200).tolist())
            + "|"
            + model_name
        )
        self.signature = hashlib.md5(signature_text.encode()).hexdigest()[:12]

        self.emb_path = self.cache_dir / f"recipe_embeddings_{self.signature}.npy"
        self.faiss_path = self.cache_dir / f"recipe_index_{self.signature}.faiss"

        self.model = None
        self.embeddings = None
        self.index = None
        self.faiss_enabled = False

    def build(self):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(self.model_name)

        if self.emb_path.exists():
            self.embeddings = np.load(self.emb_path)
            print("Loaded cached embeddings:", self.emb_path)
        else:
            self.embeddings = self.model.encode(
                self.texts,
                batch_size=64,
                show_progress_bar=True,
                normalize_embeddings=True,
            )
            self.embeddings = np.asarray(self.embeddings, dtype="float32")
            np.save(self.emb_path, self.embeddings)
            print("Saved embeddings:", self.emb_path)

        try:
            import faiss

            if self.faiss_path.exists():
                self.index = faiss.read_index(str(self.faiss_path))
                print("Loaded cached FAISS index:", self.faiss_path)
            else:
                self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
                self.index.add(self.embeddings)
                faiss.write_index(self.index, str(self.faiss_path))
                print("Saved FAISS index:", self.faiss_path)

            self.faiss_enabled = True

        except Exception as e:
            from sklearn.neighbors import NearestNeighbors

            print("FAISS unavailable; using sklearn fallback:", e)
            self.index = NearestNeighbors(metric="cosine")
            self.index.fit(self.embeddings)
            self.faiss_enabled = False

        return self

    def encode_query(self, query):
        q = self.model.encode([query], normalize_embeddings=True)
        return np.asarray(q, dtype="float32")

    def search(self, query, top_k=10):
        top_k = min(top_k, len(self.df))
        q = self.encode_query(query)

        if self.faiss_enabled:
            scores, idx = self.index.search(q, top_k)
            return idx[0], scores[0]

        distances, idx = self.index.kneighbors(q, n_neighbors=top_k)
        return idx[0], 1 - distances[0]


recipe_index = RecipeEmbeddingIndex(recipes).build()

print("Recipes indexed:", len(recipes))
print("FAISS enabled:", recipe_index.faiss_enabled)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Saved embeddings: /kaggle/working/souschef_cache/recipe_embeddings_f5cc7367ac23.npy
Saved FAISS index: /kaggle/working/souschef_cache/recipe_index_f5cc7367ac23.faiss
Recipes indexed: 20000
FAISS enabled: True


## 7. Semantic Recipe Retrieval + Cross-Encoder Reranking

FAISS gives fast candidate retrieval. An optional **cross-encoder** then scores `(user query, recipe text)` pairs more carefully.

Pipeline:

`query → embedding → FAISS top-N → CrossEncoder rerank → top results`

In [8]:
_cross_encoder = None


def get_cross_encoder():
    global _cross_encoder

    if _cross_encoder is None:
        from sentence_transformers import CrossEncoder
        print("Loading cross-encoder reranker...")
        _cross_encoder = CrossEncoder(RERANKER_MODEL)

    return _cross_encoder


def minmax_array(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    if np.nanmax(values) == np.nanmin(values):
        return np.ones(len(values)) * 0.5
    return (values - np.nanmin(values)) / (np.nanmax(values) - np.nanmin(values))


def semantic_recipe_search(query, top_k=8, candidate_k=50, use_reranker=False):
    idx, scores = recipe_index.search(query, top_k=max(top_k, candidate_k))

    result = recipes.iloc[idx].copy().reset_index(drop=True)
    result["semantic_score"] = scores

    if use_reranker and len(result):
        model = get_cross_encoder()
        pairs = [(query, t) for t in result["recipe_text"].tolist()]
        cross_scores = model.predict(pairs, batch_size=32, show_progress_bar=False)
        result["rerank_score"] = minmax_array(cross_scores)
        result = result.sort_values("rerank_score", ascending=False)
    else:
        result["rerank_score"] = minmax_array(result["semantic_score"].values)

    return result.head(top_k)


semantic_recipe_search(
    "quick vegetarian dinner with tomato garlic rice",
    top_k=5,
    use_reranker=False,
)[
    ["name", "minutes_clean", "rating", "review_count", "semantic_score", "rerank_score"]
]

,name,minutes_clean,rating,review_count,semantic_score,rerank_score
0,add an ingredient risotto,40.0,0.0000,0.0,0.662858,1.000000
1,awesomely easy marinara sauce,15.0,4.8125,16.0,0.657248,0.910892
2,2 5 challenge friendly real pasta sauce,45.0,0.0000,0.0,0.655736,0.886883
3,bacon fried rice with a side of tomatoes in fish sauce,45.0,0.0000,0.0,0.649864,0.793619
4,baked garlic tomatoes,130.0,4.0000,1.0,0.648915,0.778540


## 8. Semantic RAG Cooking Assistant

This replaces the earlier TF-IDF RAG with **SentenceTransformer + FAISS semantic retrieval**.

The knowledge base contains:
- structured cooking-fix notes
- optional `culinary_knowledge.csv`
- a sample of Food.com recipe chunks

Two answer modes are available:
1. **Local grounded answer** — no API required; shows the retrieved evidence.
2. **LLM grounded answer** — optional OpenAI key; the model is explicitly instructed to use retrieved evidence and cite `[S1]`, `[S2]`, etc.

In [9]:
CULINARY_FALLBACK = [
    {
        "source": "culinary::salty",
        "text": "For an overly salty dish, increase unsalted bulk or liquid, add starch where appropriate, and rebalance with fat or acid. Avoid trying to remove salt with sugar alone."
    },
    {
        "source": "culinary::spicy",
        "text": "To reduce excessive chile heat, add dairy when compatible, coconut milk, more base ingredients, starch, or a small amount of sweetness. Water alone can dilute flavor without balancing heat."
    },
    {
        "source": "culinary::watery_sauce",
        "text": "A watery sauce can be reduced by simmering uncovered. Depending on the dish, thicken with a starch slurry, blended vegetables, nut paste, roux, or concentrated tomato paste."
    },
    {
        "source": "culinary::split_sauce",
        "text": "A split emulsion sauce can often be rescued by lowering heat and slowly whisking the broken sauce into a small fresh base such as water, egg yolk, mustard, or cream depending on the sauce."
    },
    {
        "source": "culinary::dry_protein",
        "text": "Dry cooked protein cannot regain lost moisture internally, but serving it with sauce, broth, fat, or slicing it thinly can improve perceived juiciness. Prevent dryness by controlling final temperature."
    },
    {
        "source": "culinary::substitution_roles",
        "text": "Ingredient substitutions work best when the replacement serves the same role: fat, acid, sweetness, structure, thickening, leavening, protein, or aroma."
    },
    {
        "source": "culinary::vegan_dairy",
        "text": "Common vegan replacements include plant milks for milk, coconut cream or cashew cream for heavy cream, nutritional yeast for savory cheese notes, and plant butter for butter."
    },
    {
        "source": "culinary::baking_precision",
        "text": "Baking substitutions are less forgiving than savory cooking because flour, fat, sugar, eggs, moisture, and leavening affect structure. Change one major variable at a time where possible."
    },
]


def load_knowledge_base():
    docs = list(CULINARY_FALLBACK)

    knowledge_file = find_file(["culinary_knowledge.csv"])
    if knowledge_file:
        try:
            kb = pd.read_csv(knowledge_file)
            for _, row in kb.iterrows():
                docs.append(
                    {
                        "source": f"knowledge::{row.get('topic', 'topic')}",
                        "text": str(row.get("content", "")),
                    }
                )
        except Exception as e:
            print("Could not load culinary_knowledge.csv:", e)

    # Add recipe chunks for recipe-specific questions.
    recipe_sample = recipes.head(min(len(recipes), RAG_RECIPE_CHUNKS))
    for _, row in recipe_sample.iterrows():
        docs.append(
            {
                "source": f"recipe::{row['recipe_id']}::{row['name']}",
                "text": (
                    f"Recipe: {row['name']}. "
                    f"Ingredients: {row['ingredient_text']}. "
                    f"Instructions: {row['step_text']}. "
                    f"Tags: {row['tag_text']}."
                ),
            }
        )

    return pd.DataFrame(docs).drop_duplicates("text").reset_index(drop=True)


class SemanticKnowledgeIndex:
    def __init__(self, docs, model):
        self.docs = docs.reset_index(drop=True)
        self.model = model
        self.embeddings = None
        self.index = None
        self.faiss_enabled = False

    def build(self):
        self.embeddings = self.model.encode(
            self.docs["text"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        self.embeddings = np.asarray(self.embeddings, dtype="float32")

        try:
            import faiss
            self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
            self.index.add(self.embeddings)
            self.faiss_enabled = True
        except Exception:
            from sklearn.neighbors import NearestNeighbors
            self.index = NearestNeighbors(metric="cosine")
            self.index.fit(self.embeddings)

        return self

    def search(self, query, top_k=5):
        top_k = min(top_k, len(self.docs))
        q = self.model.encode([query], normalize_embeddings=True)
        q = np.asarray(q, dtype="float32")

        if self.faiss_enabled:
            scores, idx = self.index.search(q, top_k)
            idx, scores = idx[0], scores[0]
        else:
            distances, idx = self.index.kneighbors(q, n_neighbors=top_k)
            idx, scores = idx[0], 1 - distances[0]

        out = self.docs.iloc[idx].copy()
        out["score"] = scores
        return out.reset_index(drop=True)


rag_docs = load_knowledge_base()
rag_index = SemanticKnowledgeIndex(rag_docs, recipe_index.model).build()

print("RAG documents:", len(rag_docs))
print("RAG FAISS enabled:", rag_index.faiss_enabled)


def retrieve_knowledge(question, top_k=5):
    return rag_index.search(question, top_k=top_k)


def grounded_local_answer(question, top_k=5):
    evidence = retrieve_knowledge(question, top_k=top_k)

    lines = [
        "I retrieved the following evidence that is most relevant to your question:",
        "",
    ]

    for i, row in evidence.iterrows():
        text = row["text"]
        # Keep local mode extractive/grounded instead of pretending to be an LLM.
        short = text[:420] + ("..." if len(text) > 420 else "")
        lines.append(f"[S{i+1}] {short}")

    lines += [
        "",
        "Use these retrieved suggestions as the basis for the cooking decision.",
        "For food-safety-sensitive questions, verify safe temperatures and handling guidance from an authoritative source.",
    ]

    return "\n".join(lines), evidence


def get_secret(name):
    # Works with Kaggle Secrets if configured; otherwise environment variable.
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name)


def llm_rag_answer(question, top_k=5):
    evidence = retrieve_knowledge(question, top_k=top_k)
    api_key = get_secret("OPENAI_API_KEY")

    if not api_key:
        local, _ = grounded_local_answer(question, top_k=top_k)
        return (
            "OPENAI_API_KEY is not configured, so SousChef is showing grounded retrieval instead.\n\n"
            + local,
            evidence,
        )

    context = "\n\n".join(
        f"[S{i+1}] Source: {row.source}\n{row.text}"
        for i, row in enumerate(evidence.itertuples())
    )

    prompt = f'''
You are SousChef AI, a practical cooking copilot.

Use the retrieved evidence below as your factual basis.
Do not invent recipe facts that are absent from the evidence.
When you use a retrieved fact, cite the source marker such as [S1].
If the evidence is insufficient, clearly say what is uncertain.
For food-safety-sensitive questions, add a short safety caveat.

EVIDENCE
{context}

USER QUESTION
{question}

Return:
1. A concise direct answer.
2. Practical steps.
3. A short "Why this works" explanation.
'''

    try:
        from openai import OpenAI

        client = OpenAI(api_key=api_key)
        model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )

        return response.choices[0].message.content, evidence

    except Exception as e:
        local, _ = grounded_local_answer(question, top_k=top_k)
        return f"LLM generation failed ({e}).\n\n{local}", evidence


demo_answer, demo_evidence = grounded_local_answer(
    "My curry is too spicy. How can I reduce the heat?"
)
print(demo_answer[:1500])

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

RAG documents: 3008
RAG FAISS enabled: True
I retrieved the following evidence that is most relevant to your question:

[S1] Recipe: harak mas curry  sri lankan beef curry. Ingredients: stewing beef ghee onions fresh ginger garlic cloves curry powder ground turmeric black mustard seeds salt vinegar chilies tomatoes. Instructions: cut steak into 2 inch squares heat the ghee in a saucepan and gently fry the onions ginger and garlic until just beginning to turn golden add curry powder turmeric mustard seeds and fry over a low heat for 2-3 min...
[S2] Recipe: a ramchandani special  subru uncle s traditional sindhi curry mo. Ingredients: vegetable drumsticks ladyfingers potatoes yams beans white pumpkin carrots green chilies turmeric powder red chili powder oil tomatoes fenugreek seeds cumin seeds fresh ginger curry leaves gram flour asafoetida powder water salt tamarind pulp fresh coriander leaves. Instructions: boil the potatoes in plenty of water until tender dr...
[S3] Recipe: achingya 

## 9. Hybrid Recipe Ranking System

This section combines several independent signals:

- semantic relevance
- ingredient coverage
- **Bayesian weighted rating**
- cooking-time fit
- optional collaborative-filtering score
- optional cross-encoder reranker score
- hard diet/time/exclusion filters

### Collaborative filtering

When user interactions are available, the notebook trains a compact **matrix-factorization model** using `TruncatedSVD` on the sparse user–recipe rating matrix.

If no `user_id` is supplied, collaborative score stays neutral and recommendations remain content-based.

In [10]:
def normalize_series(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0:
        return s
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)) * 0.5, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


class CollaborativeRecipeModel:
    def __init__(
        self,
        interactions,
        allowed_recipe_ids,
        min_user_ratings=5,
        max_users=5000,
        max_items=10000,
        n_factors=32,
    ):
        self.interactions = interactions.copy()
        self.allowed_recipe_ids = set(allowed_recipe_ids)
        self.min_user_ratings = min_user_ratings
        self.max_users = max_users
        self.max_items = max_items
        self.n_factors = n_factors

        self.ready = False
        self.user_to_idx = {}
        self.item_to_idx = {}
        self.user_factors = None
        self.item_factors = None

    def fit(self):
        required = {"user_id", "recipe_id", "rating"}
        if not required.issubset(self.interactions.columns) or len(self.interactions) < 100:
            print("Collaborative filtering skipped: interactions unavailable.")
            return self

        df = self.interactions[
            self.interactions["recipe_id"].isin(self.allowed_recipe_ids)
            & (self.interactions["rating"] > 0)
        ].dropna(subset=["user_id", "recipe_id", "rating"]).copy()

        counts = df["user_id"].value_counts()
        keep_users = counts[counts >= self.min_user_ratings].head(self.max_users).index
        df = df[df["user_id"].isin(keep_users)]

        item_counts = df["recipe_id"].value_counts()
        keep_items = item_counts.head(self.max_items).index
        df = df[df["recipe_id"].isin(keep_items)]

        if df["user_id"].nunique() < 10 or df["recipe_id"].nunique() < 10:
            print("Collaborative filtering skipped: not enough dense interaction data.")
            return self

        users = df["user_id"].drop_duplicates().tolist()
        items = df["recipe_id"].drop_duplicates().tolist()

        self.user_to_idx = {u: i for i, u in enumerate(users)}
        self.item_to_idx = {r: i for i, r in enumerate(items)}

        # Center each user's ratings to model preference rather than rating scale.
        user_mean = df.groupby("user_id")["rating"].transform("mean")
        centered = df["rating"] - user_mean

        rows = df["user_id"].map(self.user_to_idx).to_numpy()
        cols = df["recipe_id"].map(self.item_to_idx).to_numpy()

        matrix = csr_matrix(
            (centered.to_numpy(), (rows, cols)),
            shape=(len(users), len(items)),
        )

        n_components = min(
            self.n_factors,
            max(2, min(matrix.shape) - 1),
        )

        svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
        self.user_factors = svd.fit_transform(matrix)
        self.item_factors = svd.components_.T
        self.ready = True

        print(
            f"Collaborative model ready: {len(users):,} users × "
            f"{len(items):,} recipes × {n_components} factors"
        )
        return self

    def score(self, user_id, recipe_ids):
        neutral = np.ones(len(recipe_ids), dtype=float) * 0.5

        if not self.ready or user_id not in self.user_to_idx:
            return neutral

        u = self.user_factors[self.user_to_idx[user_id]]
        raw = []

        for recipe_id in recipe_ids:
            item_idx = self.item_to_idx.get(recipe_id)
            if item_idx is None:
                raw.append(np.nan)
            else:
                raw.append(float(np.dot(u, self.item_factors[item_idx])))

        raw = np.asarray(raw, dtype=float)

        known = ~np.isnan(raw)
        if known.sum() < 2:
            return neutral

        scaled = neutral.copy()
        scaled[known] = minmax_array(raw[known])
        return scaled


cf_model = CollaborativeRecipeModel(
    interactions,
    recipes["recipe_id"].tolist(),
).fit()


def hybrid_recommend(
    user_ingredients_text,
    query="",
    diet="Any",
    max_minutes=60,
    excluded_ingredients_text="",
    user_id=None,
    top_k=10,
    use_reranker=False,
):
    user_ingredients = parse_user_ingredients(user_ingredients_text)
    excluded = parse_user_ingredients(excluded_ingredients_text)

    query_text = (query + " " + " ".join(user_ingredients)).strip()
    if not query_text:
        query_text = "easy recipe"

    candidate_count = min(300, len(recipes))
    idx, semantic_scores = recipe_index.search(query_text, top_k=candidate_count)

    candidates = recipes.iloc[idx].copy().reset_index(drop=True)
    candidates["semantic_score"] = semantic_scores

    # Hard filters first.
    candidates = candidates[candidates["minutes_clean"] <= max_minutes].copy()

    if diet == "Vegetarian":
        candidates = candidates[candidates["is_vegetarian"]]
    elif diet == "Vegan":
        candidates = candidates[candidates["is_vegan"]]
    elif diet == "Gluten-Free":
        candidates = candidates[candidates["is_gluten_free"]]

    if excluded:
        mask = candidates["ingredients_list"].apply(
            lambda x: not violates_exclusions(x, excluded)
        )
        candidates = candidates[mask].copy()

    if not len(candidates):
        return candidates

    overlap = candidates["ingredients_list"].apply(
        lambda x: ingredient_overlap(user_ingredients, x)
    )

    candidates["ingredient_score"] = overlap.apply(lambda x: x[0])
    candidates["matched_ingredients"] = overlap.apply(lambda x: ", ".join(x[1]))
    candidates["missing_ingredients"] = overlap.apply(
        lambda x: ", ".join(x[2][:12])
    )

    candidates["bayesian_score"] = normalize_series(candidates["bayesian_rating"])

    candidates["time_score"] = (
        1 - candidates["minutes_clean"] / max(max_minutes, 1)
    ).clip(0, 1)

    candidates["cf_score"] = cf_model.score(
        user_id,
        candidates["recipe_id"].tolist(),
    )

    candidates["rerank_score"] = normalize_series(candidates["semantic_score"])

    if use_reranker and len(candidates):
        rerank_n = min(60, len(candidates))
        head = candidates.head(rerank_n).copy()

        model = get_cross_encoder()
        pairs = [(query_text, t) for t in head["recipe_text"].tolist()]
        cross_scores = model.predict(
            pairs,
            batch_size=32,
            show_progress_bar=False,
        )

        candidates.loc[head.index, "rerank_score"] = minmax_array(cross_scores)

    # If no recognized user profile exists, redistribute CF weight.
    personalized = cf_model.ready and user_id in cf_model.user_to_idx

    if personalized:
        weights = {
            "rerank": 0.25,
            "ingredient": 0.25,
            "bayesian": 0.15,
            "time": 0.10,
            "cf": 0.25,
        }
    else:
        weights = {
            "rerank": 0.35,
            "ingredient": 0.30,
            "bayesian": 0.20,
            "time": 0.15,
            "cf": 0.00,
        }

    candidates["final_score"] = (
        weights["rerank"] * normalize_series(candidates["rerank_score"])
        + weights["ingredient"] * candidates["ingredient_score"]
        + weights["bayesian"] * candidates["bayesian_score"]
        + weights["time"] * candidates["time_score"]
        + weights["cf"] * candidates["cf_score"]
    )

    candidates["why"] = candidates.apply(
        lambda r: (
            f"{len([x for x in r['matched_ingredients'].split(', ') if x])} ingredients matched; "
            f"{r['minutes_clean']:.0f} min; "
            f"weighted rating {r['bayesian_rating']:.2f}; "
            f"semantic relevance {r['semantic_score']:.2f}"
        ),
        axis=1,
    )

    columns = [
        "recipe_id",
        "name",
        "minutes_clean",
        "rating",
        "review_count",
        "bayesian_rating",
        "final_score",
        "ingredient_score",
        "semantic_score",
        "cf_score",
        "matched_ingredients",
        "missing_ingredients",
        "why",
    ]

    return (
        candidates.sort_values("final_score", ascending=False)
        [columns]
        .head(top_k)
        .reset_index(drop=True)
    )


hybrid_recommend(
    "tomato, garlic, rice, onion",
    query="quick vegetarian dinner",
    diet="Vegetarian",
    max_minutes=45,
    top_k=5,
)

Collaborative model ready: 1,361 users × 6,689 recipes × 32 factors


,recipe_id,name,minutes_clean,rating,review_count,bayesian_rating,final_score,ingredient_score,semantic_score,cf_score,matched_ingredients,missing_ingredients,why
0,137860,awesomely easy marinara sauce,15.0,4.8125,16.0,4.723000,0.646462,0.200000,0.689959,0.5,"garlic, onion","dried basil, dried rosemary, olive oil, oregano, pepper, salt, sugar, tomato sauce",2 ingredients matched; 15 min; weighted rating 4.72; semantic relevance 0.69
1,136471,7 minute vegetable soup,30.0,0.0000,0.0,4.579801,0.544212,0.333333,0.678188,0.5,"garlic, onion","dried basil, frozen mixed vegetables, tomato sauce, tomatoes",2 ingredients matched; 30 min; weighted rating 4.58; semantic relevance 0.68
2,429838,10 minute tomato sauce from america s test kitchen,10.0,0.0000,0.0,4.579801,0.540108,0.111111,0.676911,0.5,onion,"butter, crushed tomatoes, dried oregano, fresh basil, garlic cloves, olive oil, salt, sugar",1 ingredients matched; 10 min; weighted rating 4.58; semantic relevance 0.68
3,136399,8 vegetable ragout,30.0,5.0000,1.0,4.618001,0.532115,0.083333,0.693241,0.5,onion,"carrot, celery, corn kernel, fresh parsley, hot pepper sauce, italian stewed tomatoes, minced garlic cloves, pepper, romano beans, slice...",1 ingredients matched; 30 min; weighted rating 4.62; semantic relevance 0.69
4,148362,2 5 challenge friendly real pasta sauce,45.0,0.0000,0.0,4.579801,0.515799,0.090909,0.707529,0.5,onion,"chopped tomatoes, courgette, dried herbs, garlic clove, mushrooms, olive oil, paprika, pepper, salt and pepper, sweetcorn",1 ingredients matched; 45 min; weighted rating 4.58; semantic relevance 0.71


## 10. Cuisine Classification Model

Attach Kaggle's **What's Cooking?** dataset if you want this section to run.

Instead of training one model and reporting one accuracy number, SousChef compares:
- TF-IDF + Logistic Regression
- TF-IDF + Linear SVM

The winner is selected by **Macro-F1**, which is more informative when cuisine classes are imbalanced.

In [11]:
def load_whats_cooking_if_available():
    train_json = find_file(["train.json"])

    if train_json is None:
        return None

    try:
        wc = pd.read_json(train_json)

        if not {"cuisine", "ingredients"}.issubset(wc.columns):
            return None

        wc["ingredient_text"] = wc["ingredients"].apply(
            lambda x: " ".join(clean_list_items(x))
        )

        return wc[["cuisine", "ingredient_text"]].dropna()

    except Exception as e:
        print("Could not load What's Cooking dataset:", e)
        return None


cuisine_df = load_whats_cooking_if_available()
cuisine_model = None
cuisine_results = pd.DataFrame()
cuisine_test = None

if cuisine_df is None or cuisine_df["cuisine"].nunique() < 2:
    print(
        "Cuisine classifier skipped. Add the Kaggle 'What's Cooking?' "
        "dataset so train.json is available."
    )

else:
    cuisine_df = cuisine_df.sample(
        min(len(cuisine_df), 30_000),
        random_state=RANDOM_STATE,
    )

    X_train, X_test, y_train, y_test = train_test_split(
        cuisine_df["ingredient_text"],
        cuisine_df["cuisine"],
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=cuisine_df["cuisine"],
    )

    candidate_models = {
        "TFIDF + LogisticRegression": Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    max_features=30_000,
                    ngram_range=(1, 2),
                    stop_words="english",
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=1200,
                    n_jobs=-1,
                    class_weight="balanced",
                ),
            ),
        ]),
        "TFIDF + LinearSVC": Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    max_features=30_000,
                    ngram_range=(1, 2),
                    stop_words="english",
                ),
            ),
            (
                "clf",
                LinearSVC(class_weight="balanced"),
            ),
        ]),
    }

    result_rows = []
    fitted = {}

    for name, model in candidate_models.items():
        print("Training:", name)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        acc = accuracy_score(y_test, pred)
        macro_f1 = f1_score(y_test, pred, average="macro")

        result_rows.append(
            {
                "model": name,
                "accuracy": acc,
                "macro_f1": macro_f1,
            }
        )
        fitted[name] = model

    cuisine_results = pd.DataFrame(result_rows).sort_values(
        "macro_f1",
        ascending=False,
    )

    best_name = cuisine_results.iloc[0]["model"]
    cuisine_model = fitted[best_name]
    cuisine_test = (X_test, y_test)

    print("\nModel comparison:")
    display(cuisine_results)

    best_pred = cuisine_model.predict(X_test)
    print("\nBest model:", best_name)
    print(classification_report(y_test, best_pred, zero_division=0))


def predict_cuisine_from_ingredients(ingredients_text):
    if cuisine_model is None:
        return (
            "Cuisine model unavailable. Attach the Kaggle "
            "'What's Cooking?' dataset."
        )

    clean_text = " ".join(parse_user_ingredients(ingredients_text))
    pred = cuisine_model.predict([clean_text])[0]

    return pred


predict_cuisine_from_ingredients(
    "tomato, garlic, cumin, coriander, chickpeas"
)

Training: TFIDF + LogisticRegression
Training: TFIDF + LinearSVC

Model comparison:


,model,accuracy,macro_f1
1,TFIDF + LinearSVC,0.776333,0.695604
0,TFIDF + LogisticRegression,0.746667,0.672436



Best model: TFIDF + LinearSVC
              precision    recall  f1-score   support

   brazilian       0.60      0.60      0.60        68
     british       0.54      0.54      0.54       123
cajun_creole       0.72      0.79      0.75       238
     chinese       0.77      0.84      0.80       401
    filipino       0.62      0.58      0.60       110
      french       0.61      0.63      0.62       409
       greek       0.70      0.72      0.71       178
      indian       0.86      0.89      0.87       461
       irish       0.50      0.54      0.52       104
     italian       0.87      0.84      0.85      1182
    jamaican       0.72      0.78      0.75        74
    japanese       0.78      0.66      0.72       215
      korean       0.75      0.73      0.74       125
     mexican       0.92      0.91      0.92       963
    moroccan       0.77      0.85      0.81       121
     russian       0.48      0.43      0.46        72
 southern_us       0.76      0.75      0.75       

'indian'

## 11. Evaluation

### A. Semantic recipe retrieval

For sampled recipes, the notebook hides part of the recipe and creates a query from a subset of its ingredients. It then checks whether the original recipe is retrieved.

Metrics:
- HitRate@1 / @5 / @10
- MRR (Mean Reciprocal Rank)

### B. RAG retrieval

A small ground-truth cooking set checks whether evidence containing the expected concepts appears in the top retrieved documents.

This is much stronger than evaluating the recommender using its own internal score.

In [12]:
def evaluate_semantic_retrieval(n_samples=100, ks=(1, 5, 10)):
    eligible = recipes[recipes["ingredient_count"] >= 4]

    if len(eligible) == 0:
        return pd.DataFrame(), {}

    sample = eligible.sample(
        min(n_samples, len(eligible)),
        random_state=RANDOM_STATE,
    )

    ranks = []

    for _, row in sample.iterrows():
        ingredients = row["ingredients_list"]
        keep_n = max(2, int(len(ingredients) * 0.60))
        query_ingredients = ingredients[:keep_n]

        query = "recipe with " + ", ".join(query_ingredients)

        idx, _ = recipe_index.search(query, top_k=max(ks))
        retrieved_ids = recipes.iloc[idx]["recipe_id"].tolist()

        try:
            rank = retrieved_ids.index(row["recipe_id"]) + 1
        except ValueError:
            rank = None

        ranks.append(rank)

    metrics = {}

    for k in ks:
        metrics[f"HitRate@{k}"] = np.mean(
            [r is not None and r <= k for r in ranks]
        )

    metrics["MRR"] = np.mean(
        [1 / r if r is not None else 0 for r in ranks]
    )

    detail = pd.DataFrame({"rank": ranks})
    return detail, metrics


RAG_EVAL_SET = [
    {
        "question": "How can I fix a sauce that is too watery?",
        "expected_terms": ["watery", "simmer", "slurry"],
    },
    {
        "question": "My curry is much too spicy. What can I do?",
        "expected_terms": ["spicy", "dairy", "coconut"],
    },
    {
        "question": "How do I rescue an overly salty dish?",
        "expected_terms": ["salty", "unsalted", "starch"],
    },
    {
        "question": "What should I consider when substituting ingredients in baking?",
        "expected_terms": ["baking", "structure", "leavening"],
    },
]


def evaluate_rag_retrieval(eval_set=RAG_EVAL_SET, top_k=5):
    rows = []

    for item in eval_set:
        evidence = retrieve_knowledge(item["question"], top_k=top_k)
        expected = [normalize_text(x) for x in item["expected_terms"]]

        rank = None

        for i, text in enumerate(evidence["text"].tolist(), start=1):
            t = normalize_text(text)

            # Ground-truth document counts as found when it contains at least
            # two of the expected concepts.
            matches = sum(term in t for term in expected)

            if matches >= min(2, len(expected)):
                rank = i
                break

        rows.append(
            {
                "question": item["question"],
                "rank": rank,
                f"hit@{top_k}": rank is not None,
                "reciprocal_rank": 1 / rank if rank else 0,
            }
        )

    df = pd.DataFrame(rows)

    summary = {
        f"HitRate@{top_k}": df[f"hit@{top_k}"].mean(),
        "MRR": df["reciprocal_rank"].mean(),
    }

    return df, summary


retrieval_eval_detail, retrieval_metrics = evaluate_semantic_retrieval()
rag_eval_detail, rag_metrics = evaluate_rag_retrieval()

print("Semantic retrieval metrics:")
display(pd.DataFrame([retrieval_metrics]))

print("RAG retrieval metrics:")
display(pd.DataFrame([rag_metrics]))

display(rag_eval_detail)

Semantic retrieval metrics:


,HitRate@1,HitRate@5,HitRate@10,MRR
0,0.26,0.44,0.47,0.333583


RAG retrieval metrics:


,HitRate@5,MRR
0,1.0,0.833333


,question,rank,hit@5,reciprocal_rank
0,How can I fix a sauce that is too watery?,1,True,1.000000
1,My curry is much too spicy. What can I do?,3,True,0.333333
2,How do I rescue an overly salty dish?,1,True,1.000000
3,What should I consider when substituting ingredients in baking?,1,True,1.000000


## 12. Notebook Dashboard

The notebook dashboard is intentionally kept inside Jupyter/Kaggle.

Tabs:
- 🍳 Recipe Finder
- 🤖 Ask SousChef
- 🌍 Cuisine Predictor
- 📊 Evaluation

This fixes the earlier `VBox` issue by using `widgets.HTML` rather than `IPython.display.HTML` as a widget child.

In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output


# ---------- Recipe Finder ----------
finder_ingredients = widgets.Textarea(
    value="tomato, garlic, rice, onion",
    description="Ingredients:",
    placeholder="tomato, garlic, rice, onion",
    layout=widgets.Layout(width="100%", height="90px"),
)

finder_goal = widgets.Text(
    value="quick vegetarian dinner",
    description="Goal:",
    layout=widgets.Layout(width="100%"),
)

finder_diet = widgets.Dropdown(
    options=["Any", "Vegetarian", "Vegan", "Gluten-Free"],
    value="Any",
    description="Diet:",
)

finder_time = widgets.IntSlider(
    value=45,
    min=10,
    max=180,
    step=5,
    description="Max min:",
    continuous_update=False,
)

finder_exclude = widgets.Text(
    value="",
    description="Exclude:",
    placeholder="peanut, shellfish, mushrooms...",
    layout=widgets.Layout(width="100%"),
)

finder_user = widgets.Text(
    value="",
    description="User ID:",
    placeholder="Optional Food.com user_id",
)

finder_rerank = widgets.Checkbox(
    value=False,
    description="Use cross-encoder reranker",
)

finder_button = widgets.Button(
    description="Find Recipes",
    button_style="success",
    icon="search",
)

finder_output = widgets.Output()


def run_finder(_):
    with finder_output:
        clear_output()

        try:
            user_id = int(finder_user.value) if finder_user.value.strip() else None
        except ValueError:
            user_id = finder_user.value.strip() or None

        recs = hybrid_recommend(
            finder_ingredients.value,
            query=finder_goal.value,
            diet=finder_diet.value,
            max_minutes=finder_time.value,
            excluded_ingredients_text=finder_exclude.value,
            user_id=user_id,
            top_k=8,
            use_reranker=finder_rerank.value,
        )

        if not len(recs):
            print("No recipes matched. Broaden the filters.")
            return

        show = recs.copy()
        show["final_score"] = show["final_score"].round(3)
        show["bayesian_rating"] = show["bayesian_rating"].round(2)

        display(
            show[
                [
                    "name",
                    "minutes_clean",
                    "bayesian_rating",
                    "final_score",
                    "matched_ingredients",
                    "missing_ingredients",
                    "why",
                ]
            ]
        )


finder_button.on_click(run_finder)

finder_tab = widgets.VBox([
    widgets.HTML(
        "<h3>🍳 Recipe Finder</h3>"
        "<p>Semantic retrieval + ingredient matching + weighted rating "
        "+ optional personalization.</p>"
    ),
    finder_ingredients,
    finder_goal,
    widgets.HBox([finder_diet, finder_time]),
    finder_exclude,
    widgets.HBox([finder_user, finder_rerank]),
    finder_button,
    finder_output,
])


# ---------- RAG Assistant ----------
rag_question = widgets.Textarea(
    value="My curry is too spicy. How can I fix it?",
    description="Question:",
    layout=widgets.Layout(width="100%", height="100px"),
)

rag_mode = widgets.Dropdown(
    options=["Grounded local retrieval", "LLM RAG"],
    value="Grounded local retrieval",
    description="Mode:",
)

rag_button = widgets.Button(
    description="Ask SousChef",
    button_style="primary",
)

rag_output = widgets.Output()


def run_rag(_):
    with rag_output:
        clear_output()

        if rag_mode.value == "LLM RAG":
            answer, evidence = llm_rag_answer(rag_question.value)
        else:
            answer, evidence = grounded_local_answer(rag_question.value)

        print(answer)

        print("\nTop evidence:")
        display(evidence[["source", "score", "text"]].head(5))


rag_button.on_click(run_rag)

rag_tab = widgets.VBox([
    widgets.HTML(
        "<h3>🤖 Ask SousChef</h3>"
        "<p>Semantic FAISS retrieval with visible supporting evidence.</p>"
    ),
    rag_question,
    rag_mode,
    rag_button,
    rag_output,
])


# ---------- Cuisine ----------
cuisine_input = widgets.Textarea(
    value="tomato, garlic, cumin, coriander, chickpeas",
    description="Ingredients:",
    layout=widgets.Layout(width="100%", height="90px"),
)

cuisine_button = widgets.Button(
    description="Predict Cuisine",
    button_style="info",
)

cuisine_output = widgets.Output()


def run_cuisine(_):
    with cuisine_output:
        clear_output()
        print(predict_cuisine_from_ingredients(cuisine_input.value))


cuisine_button.on_click(run_cuisine)

cuisine_tab = widgets.VBox([
    widgets.HTML(
        "<h3>🌍 Cuisine Predictor</h3>"
        "<p>Runs when the What's Cooking? train.json dataset is attached.</p>"
    ),
    cuisine_input,
    cuisine_button,
    cuisine_output,
])


# ---------- Evaluation ----------
eval_button = widgets.Button(
    description="Show Evaluation",
    button_style="warning",
)

eval_output = widgets.Output()


def run_eval(_):
    with eval_output:
        clear_output()

        print("Semantic recipe retrieval")
        display(pd.DataFrame([retrieval_metrics]))

        print("\nRAG retrieval")
        display(pd.DataFrame([rag_metrics]))

        if len(cuisine_results):
            print("\nCuisine model comparison")
            display(cuisine_results)
        else:
            print("\nCuisine evaluation unavailable until train.json is attached.")


eval_button.on_click(run_eval)

eval_tab = widgets.VBox([
    widgets.HTML(
        "<h3>📊 Model Evaluation</h3>"
        "<p>Offline retrieval and classification metrics.</p>"
    ),
    eval_button,
    eval_output,
])


dashboard = widgets.Tab(
    children=[finder_tab, rag_tab, cuisine_tab, eval_tab]
)

for i, title in enumerate([
    "🍳 Recipe Finder",
    "🤖 Ask SousChef",
    "🌍 Cuisine",
    "📊 Evaluation",
]):
    dashboard.set_title(i, title)


display(
    widgets.VBox([
        widgets.HTML(
            "<h2>🥘 SousChef AI</h2>"
            "<p>RAG + FAISS + Sentence Transformers + Hybrid Recommendation</p>"
        ),
        dashboard,
    ])
)

## 13. Results and Limitations

### What this notebook now demonstrates

- real Food.com recipe and interaction data
- semantic recipe embeddings
- FAISS vector retrieval
- cached vector indexes
- optional cross-encoder reranking
- semantic RAG instead of TF-IDF-only retrieval
- grounded evidence display
- optional LLM answer generation
- Bayesian review-aware ranking
- collaborative filtering
- hybrid recommendation
- cuisine model comparison
- offline retrieval/RAG metrics
- interactive notebook dashboard

### Limitations

1. Food.com behavior is historical and may not represent current cooking preferences.
2. Dietary detection uses ingredient-text heuristics and must **not** be treated as allergy-safe.
3. Collaborative filtering is intentionally compact for Kaggle runtime; a production system would use stronger train/validation splits and dedicated recommender tooling.
4. Recipe retrieval evaluation is a proxy task based on masked ingredients.
5. The local RAG mode is extractive. Natural synthesized explanations require an LLM.
6. LLM answers can still be wrong; retrieved evidence should remain visible.
7. Food safety requires authoritative temperature/handling guidance.
8. For final reporting, record actual metrics from a clean `Restart & Run All` session rather than hard-coding performance claims.

## 14. Resume Summary

### Project title
**SousChef AI — RAG-Based Personalized Cooking Copilot**

### Resume bullets

- Built a RAG-based culinary copilot over large-scale Food.com recipe and interaction data, using **SentenceTransformer embeddings, FAISS vector search, cross-encoder reranking, and grounded LLM generation** to retrieve and explain relevant recipes and cooking guidance.

- Developed a **hybrid recommendation pipeline** combining semantic relevance, ingredient coverage, Bayesian review-weighted ratings, time/diet constraints, and matrix-factorization-based collaborative signals; added offline **HitRate@K/MRR** retrieval evaluation and cuisine classification model comparison in an interactive Kaggle/Jupyter dashboard.

> Replace qualitative wording with your actual final metrics after running the notebook end-to-end.

## 15. Export for the Streamlit dashboard
After training/evaluation, run the next cell to package your actual trained assets for Streamlit.

In [ ]:
# PASTE THIS CELL AT THE END of souschef-1.ipynb, AFTER all model and evaluation cells.
# The variables below come from your existing SousChef AI v2 Kaggle notebook.
# This will export the exact trained models (no model retraining in Streamlit).
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import joblib
import faiss

needed = ["recipes", "recipe_index", "rag_docs", "rag_index", "cf_model"]
missing = [v for v in needed if v not in globals()]
if missing:
    raise RuntimeError(f"Run the training notebook first; missing variables: {missing}")

output_dir = Path("/kaggle/working/souschef_streamlit_artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

# Preserve exact row ordering matching the saved FAISS indexes.
recipes.reset_index(drop=True).to_parquet(output_dir / "recipes.parquet", index=False)
rag_docs.reset_index(drop=True).to_parquet(output_dir / "rag_docs.parquet", index=False)

def export_index(obj, destination):
    vectors = np.asarray(obj.embeddings, dtype=np.float32)
    if vectors.ndim != 2 or len(vectors) == 0:
        raise RuntimeError(f"No embeddings available for {destination}. Run embedding cell first.")
    # all-MiniLM-L6-v2 embeddings were normalized in the training notebook.
    faiss.normalize_L2(vectors)
    idx = faiss.IndexFlatIP(vectors.shape[1])
    idx.add(vectors)
    faiss.write_index(idx, str(output_dir / destination))

export_index(recipe_index, "recipe_index.faiss")
export_index(rag_index, "rag_index.faiss")

# A saved sklearn Pipeline can be loaded at inference time; only load trusted artifacts.
if globals().get("cuisine_model") is not None:
    joblib.dump(cuisine_model, output_dir / "cuisine_model.joblib")

if getattr(cf_model, "ready", False):
    users = sorted(cf_model.user_to_idx, key=cf_model.user_to_idx.get)
    items = sorted(cf_model.item_to_idx, key=cf_model.item_to_idx.get)
    np.savez_compressed(
        output_dir / "collab_factors.npz",
        user_factors=np.asarray(cf_model.user_factors, dtype=np.float32),
        item_factors=np.asarray(cf_model.item_factors, dtype=np.float32),
        users=np.asarray([str(x) for x in users]),
        items=np.asarray([str(x) for x in items]),
    )

metrics_export = {}
for name, variable in [("semantic_retrieval", "retrieval_metrics"), ("rag_retrieval", "rag_metrics")]:
    raw = globals().get(variable, {})
    if isinstance(raw, dict):
        metrics_export[name] = {k: float(v) for k, v in raw.items()}
(output_dir / "metrics.json").write_text(json.dumps(metrics_export, indent=2))

if isinstance(globals().get("cuisine_results"), pd.DataFrame) and not cuisine_results.empty:
    cuisine_results.to_csv(output_dir / "cuisine_results.csv", index=False)

manifest = {
    "source": "SousChef AI v2 Kaggle notebook",
    "recipe_count": len(recipes),
    "rag_document_count": len(rag_docs),
    "embedding_model": globals().get("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"),
    "has_cuisine_classifier": (output_dir / "cuisine_model.joblib").exists(),
    "has_collaborative_filtering": (output_dir / "collab_factors.npz").exists(),
    "notes": "All saved FAISS indexes and dataframes share row order. Metrics are from the notebook, not calculated in Streamlit.",
}
(output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

archive = Path("/kaggle/working/souschef_streamlit_artifacts.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in output_dir.iterdir():
        if path.is_file():
            z.write(path, arcname=f"artifacts/{path.name}")

print("Export finished:", archive)
print("Files:", [(p.name, round(p.stat().st_size / 1024**2, 2)) for p in output_dir.iterdir()])
print("Download souschef_streamlit_artifacts.zip from Kaggle's Output panel.")
